# 10 Complete Workflow (Kaggle)

End-to-end pipeline: OTLP environment setup, server preset launch,
instrumented client, and inference — all in one notebook.

**What you will learn:**
- Load Grafana OTLP credentials from Kaggle Secrets
- Start llama-server from a preset
- Create an instrumented client with OpenTelemetry
- Run inference with full observability

**Requirements:** Kaggle T4 x2. Optional: Grafana Cloud OTLP secrets.

## 1) Install

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

## 2) Load OTLP environment from Kaggle Secrets

Reads `GRAFANA_OTLP_ENDPOINT` and `GRAFANA_OTLP_HEADERS` (or `OTLP_ENDPOINT`
/ `OTLP_TOKEN`) from Kaggle Secrets and sets them as environment variables.

In [ ]:
from llamatelemetry.kaggle.pipeline import (
    KagglePipelineConfig,
    load_grafana_otlp_env_from_kaggle,
    start_server_from_preset,
    setup_otel_and_client,
)
from llamatelemetry.kaggle import ServerPreset

otlp_env = load_grafana_otlp_env_from_kaggle()
print(f"OTLP configured: {bool(otlp_env)}")

## 3) Start llama-server from a preset

In [ ]:
model_path = "/kaggle/input/your-model/model.gguf"

manager = start_server_from_preset(model_path, ServerPreset.KAGGLE_DUAL_T4)
print(f"Server started: {manager.check_server_health()}")

## 4) Setup OpenTelemetry + instrumented client

`setup_otel_and_client()` returns a dict with:
- `client` — `InstrumentedLlamaCppClient`
- `tracer` — OpenTelemetry tracer
- `meter` — OpenTelemetry meter

In [ ]:
cfg = KagglePipelineConfig(
    enable_graphistry=False,
    enable_llama_metrics=True,
)

ctx = setup_otel_and_client("http://127.0.0.1:8080", cfg)
client = ctx["client"]
print(f"Pipeline context keys: {list(ctx.keys())}")

## 5) Run inference with telemetry

In [ ]:
resp = client.chat_completions({
    "model": "local-gguf",
    "messages": [{"role": "user", "content": "Hello from the complete workflow!"}],
    "max_tokens": 64,
})
print(resp.choices[0].message.content)

## 6) Check server metrics

In [ ]:
import json

health = manager.get_health()
print(json.dumps(health, indent=2, default=str))

## 7) Cleanup

In [ ]:
manager.stop_server()
print("Done.")